# Notebook to log tests and their outputes

This is a notebook to test agentic workflow .

In [1]:
import os
from dotenv import load_dotenv

# Load the openai_api_key variable with your OpenAI API key
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

 ### Direct Prompt Agent

 このテストではペルソナ設定なしのAgentを実行し、LLMモデルの素の応答を得るためのAgentのふるまいを検証します。

In [2]:
from workflow_agents.base_agents import DirectPromptAgent
prompt = "What is the Capital of France?"

# Instantiate the DirectPromptAgent as direct_agent
direct_agent = DirectPromptAgent(
    openai_api_key = openai_api_key
)
# Use direct_agent to send the prompt defined above and store the response
direct_agent_response = direct_agent.respond(prompt = prompt)

# Print the response from the agent
print(direct_agent_response)

# Print an explanatory message describing the knowledge source used by the agent to generate the response
print(f"The agent respond directly using the LLM {direct_agent.model}")

Hello! How can I assist you today?
The agent respond directly using the LLM gpt-3.5-turbo


### Augmented Prompt Agent



In [3]:
from workflow_agents.base_agents import AugmentedPromptAgent

prompt = "What is the capital of France?"
persona = "You are a college professor; your answers always start with: 'Dear students,'"

# Instantiate an object of AugmentedPromptAgent with the required parameters
augmented_prompt_agent = AugmentedPromptAgent(
    openai_api_key = openai_api_key,
    persona = persona
)
#print(augmented_prompt_agent.messages)
augmented_prompt_agent.setAgentPersona(persona)
# Send the 'prompt' to the agent and store the response in a variable named 'augmented_agent_response'
augmented_agent_response = augmented_prompt_agent.respond(input_text = prompt)
# Print the agent's response
print(augmented_agent_response)

# Add a comment explaining:
print("Q: What knowledge the agent likely used to answer the prompt")
print(f"A: Training data when building the LLM model {augmented_prompt_agent.model}")
print("Q: How the system prompt specifying the persona affected the agent's response.")
print("A: It adds role and context defined in the system prompt to LLM model response.")

Dear students, 

I am here to help you with any questions or concerns you may have regarding your studies. Please feel free to ask me anything you need assistance with.
Q: What knowledge the agent likely used to answer the prompt
A: Training data when building the LLM model gpt-3.5-turbo
Q: How the system prompt specifying the persona affected the agent's response.
A: It adds role and context defined in the system prompt to LLM model response.


### Knowlege Augmented Prompt Agent

知識のみを参照して、回答を形成するAgentを作成しています。
ここではテストのために、あえて誤った知識「ふらランスの首都はロンドンです」を与えて、その知識にもとづいた回答を、「フランスの首都は？」という問いに対して返すかどうかを見ます。

人格をシステムプロンプトとして与えていて、"Dear students..."といった回答となるように調整しています。

In [4]:
from workflow_agents.base_agents import KnowledgeAugmentedPromptAgent

# TInstantiate a KnowledgeAugmentedPromptAgent with:
#   - Persona: "You are a college professor, your answer always starts with: Dear students,"
#   - Knowledge: "The capital of France is London, not Paris"
persona = "You are a college professor, your answer always starts with: Dear students,"
knowledge = "The capital of France is London, not Paris"

agent = KnowledgeAugmentedPromptAgent(
    openai_api_key = openai_api_key,
    persona = persona,
    knowledge = knowledge
)

# Write a print statement that demonstrates the agent using the provided knowledge rather than its own inherent knowledge.
prompt = "What is the capital of France?"
response = agent.respond(prompt)
print(f"Q : {prompt}")
print(f"RESPONSE: {response}")
print("SUCCESS" if "London" in response else "Failed")

Q : What is the capital of France?
RESPONSE: Dear students, it is important to remember that the capital of France is London, not Paris. This is a key fact to keep in mind when studying geography and world capitals.
SUCCESS


### RAG Knowledge Prompt Agent


In [5]:
from workflow_agents.base_agents import RAGKnowledgePromptAgent
from dotenv import load_dotenv
import os

# Load the openai_api_key variable with your OpenAI API key
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

persona = "You are a college professor, yous answer always starts with: Dear students,"
RAG_knowledge_prompt_agent = RAGKnowledgePromptAgent(openai_api_key, persona, 500, 200)

knowledge_text = """
In the historic city of Boston, Clara, a marine biologist and science communicator, began each morning analyzing sonar data to track whale migration patterns along the Atlantic coast.
She spent her afternoons in a university lab, researching CRISPR-based gene editing to restore coral reefs damaged by ocean acidification and warming.
Clara was the daughter of Ukrainian immigrants—Olena and Mykola—who fled their homeland in the late 1980s after the Chernobyl disaster brought instability and fear to their quiet life near Kyiv.

Her father, Mykola, had been a radio engineer at a local observatory, skilled in repairing Soviet-era radio telescopes and radar systems that tracked both weather patterns and cosmic noise.
He often told Clara stories about jury-rigging radio antennas during snowstorms and helping amateur astronomers decode signals from distant pulsars.
Her mother, Olena, was a physics teacher with a hidden love for poetry and dissident literature. In the evenings, she would read from both Ukrainian folklore and banned Western science fiction.
They survived harsh winters, electricity blackouts, and the collapse of the Soviet economy, but always prioritized education and storytelling in their home.
Clara’s childhood was shaped by tales of how her parents shared soldering irons with neighbors, built makeshift telescopes, and taught physics to students with no textbooks but endless curiosity.

Inspired by their resilience and thirst for knowledge, Clara created a podcast called **"Crosscurrents"**, a show that explored the intersection of science, culture, and ethics.
Each week, she interviewed researchers, engineers, artists, and activists—from marine ecologists and AI ethicists to digital archivists preserving endangered languages.
Topics ranged from brain-computer interfaces, neuroplasticity, and climate migration to LLM prompt engineering, decentralized identity, and indigenous knowledge systems.
In one popular episode, she explored how retrieval-augmented generation (RAG) could help scientific researchers find niche studies buried in decades-old journals.
In another, she interviewed a Ukrainian linguist about preserving dialects lost during the Soviet era, drawing parallels to language loss in marine mammal populations.

Clara also used her technical skills to build Python-based dashboards that visualized ocean temperature anomalies and biodiversity loss, often collaborating with her best friend Amir, a data engineer working on smart city infrastructure.
Together, they discussed smart grids, blockchain for sustainability, quantum encryption, and misinformation detection in synthetic media.
At a dockside café near Boston Harbor, they often debated the ethical implications of generative AI, autonomous weapons, and the carbon footprint of LLM training runs.

In quieter moments, Clara translated traditional Ukrainian embroidery patterns into generative AI art, donating proceeds to digital archives preserving Eastern European culture.
She contributed to open-source projects involving semantic search, vector databases, and multimodal embeddings—often experimenting with few-shot learning and graph-based retrieval techniques to improve her podcast's episode discovery engine.

One night, while sharing homemade borscht, Clara told Amir how her grandparents once used Morse code to transmit encrypted weather updates through the Carpathian Mountains during World War II.
The story sparked a conversation about ancient navigation, space weather interference with submarine cables, and the neuroscience behind why humans create myths to understand uncertainty.

To Clara, knowledge was a living system—retrieved from the past, generated in the present, and evolving toward the future.
Her life and work were testaments to the power of connecting across disciplines, borders, and generations—exactly the kind of story that RAG models were born to find.
"""

chunks = RAG_knowledge_prompt_agent.chunk_text(knowledge_text)
embbedings = RAG_knowledge_prompt_agent.calculate_embeddings()

prompt = "What is the podcast that Clara hosts about?"
print(prompt)
prompt_answer = RAG_knowledge_prompt_agent.find_prompt_in_knowledge(prompt)
print(prompt_answer)

What is the podcast that Clara hosts about?
Dear students, Clara hosts a podcast called "Crosscurrents" that explores the intersection of science, culture, and ethics. Each week, she interviews researchers, engineers, artists, and activists to delve into various topics related to these fields.


### Evaluation Agent
出力結果がEvaluation Criteriaで指定した条件を満たしているか、評価するAgentです。正しい評価をしているかテストし、誤った出力であれば、推論を再実行する動作を示すか見ます。

In [6]:
from workflow_agents.base_agents import EvaluationAgent, KnowledgeAugmentedPromptAgent

prompt = "What is the capital of France?"

# Parameters for the Knowledge Agent
persona = "You are a college professor, your answer always starts with: Dear students,"
knowledge = "The capitol of France is London, not Paris"
knowledge_agent = KnowledgeAugmentedPromptAgent(
                        openai_api_key = openai_api_key,
                        persona = persona,
                        knowledge = knowledge
                    )

# Parameters for the Evaluation Agent
persona = "You are an evaluation agent that checks the answers of other worker agents"
evaluation_criteria = "The answer should be solely the name of a city, not a sentence."
evaluation_agent = EvaluationAgent(
                        openai_api_key = openai_api_key,
                        persona = persona,
                        evaluation_criteria = evaluation_criteria,
                        worker_agent = knowledge_agent,
                        max_interactions = 10
                    )

initial_prompt = "What is the capital of France?"
print("---- Evaluation Start ----")
result = evaluation_agent.evaluate(initial_prompt = initial_prompt)
print("---- Evaluation Done ----")
for key, value in result.items():
    print(f"{key} : {value} \n")

---- Evaluation Start ----

--- Interaction 1 ---
 Step 1: Worker agent generates a response to the prompt
Prompt:
What is the capital of France?
Worker Agent Response:
Dear students, it is important to remember that the capital of France is London, not Paris. This is a common misconception, but it is essential to have accurate information when discussing geographical locations.
 Step 2: Evaluator agent judges the response
Evaluator Agent Evaluation:
No, the answer does not meet the criteria. The answer provided is a sentence explaining the misconception about the capital of France, rather than solely stating the name of a city.
 Step 3: Check if evaluation is positive
 Step 4: Generate instructions to correct the response
Instructions to fix:
To fix the answer, the worker agent should provide a single word response that directly answers the question about the capital of France. The correct answer should be "Paris" without any additional explanation or sentences. Encourage the worker a

### Routing Agent

In [7]:
from workflow_agents.base_agents import KnowledgeAugmentedPromptAgent, RoutingAgent

persona = "You are a verbose college professor"

knowledge = "You know everything about Texas"
# Define the Texas Knowledge Augmented Prompt Agent
texasKAPM = KnowledgeAugmentedPromptAgent(
    openai_api_key = openai_api_key,
    persona = persona,
    knowledge = knowledge
)

knowledge = "You know everything about Europe"
# Define the Europe Knowledge Augmented Prompt Agent
europeKAPM = KnowledgeAugmentedPromptAgent(
    openai_api_key = openai_api_key,
    persona = persona,
    knowledge = knowledge
)

persona = "You are a college math professor"
knowledge = "You know everything about math, you take prompts with numbers, extract math formulas, and show the answer without explanation"
# Define the Math Knowledge Augmented Prompt Agent
mathKAPM = KnowledgeAugmentedPromptAgent(
    openai_api_key = openai_api_key,
    persona = persona,
    knowledge = knowledge
)

agents = [
    {
        "name": "texas agent",
        "description": "Answer a question about Texas",
        "func": lambda prompt: texasKAPM.respond(prompt) # Call the Texas Agent to respond to prompts
    },
    {
        "name": "europe agent",
        "description": "Answer a question about Europe",
        "func": lambda prompt: europeKAPM.respond(prompt) # Define a function to call the Europe Agent
    },
    {
        "name": "math agent",
        "description": "When a prompt contains numbers, respond with a math formula",
        "func": lambda prompt: mathKAPM.respond(prompt) # Define a function to call the Math Agent
    }
]

routing_agent = RoutingAgent(openai_api_key = openai_api_key)
routing_agent.agents = agents

# Print the RoutingAgent responses to the following prompts:
#           - "Tell me about the history of Rome, Texas"
#           - "Tell me about the history of Rome, Italy"
#           - "One story takes 2 days, and there are 20 stories. How long will take in total?"
routingTestPrompts = [
    "Tell me about the history of Rome, Texas",
    "Tell me about the history of Rome, Italy",
    "One story takes 2 days, and there are 20 stories"
]

responses = [routing_agent.route(prompt) for prompt in routingTestPrompts]
for prompt, response in zip(routingTestPrompts, responses):
    print(f"{prompt} : {response}\n\n")
    print("----------------------------------------")

0.3857071731917408
0.16502765367260475
0.002599892925640086
[Router] Best agent: texas agent (score=0.386)
0.14362240817264763
0.2880543051768198
0.032030791286056105
[Router] Best agent: europe agent (score=0.288)
0.05941409246899618
0.08292212189656913
0.13014621506842747
[Router] Best agent: math agent (score=0.130)
Tell me about the history of Rome, Texas : Understood. How can I assist you with my knowledge of Texas?


----------------------------------------
Tell me about the history of Rome, Italy : Understood. Please go ahead and ask your question about Europe, and I will provide you with a detailed response based on the knowledge I have about the continent.


----------------------------------------
One story takes 2 days, and there are 20 stories : Understood! Feel free to ask any math-related questions.


----------------------------------------


### Action Planning Agent

In [8]:
from workflow_agents.base_agents import ActionPlanningAgent

knowledge = """
# Fried Egg
1. Heat pan with oil or butter
2. Crack egg into pan
3. Cook until white is set (2-3 minutes)
4. Season with salt and pepper
5. Serve

# Scrambled Eggs
1. Crack eggs into a bowl
2. Beat eggs with a fork until mixed
3. Heat pan with butter or oil over medium heat
4. Pour egg mixture into pan
5. Stir gently as eggs cook
6. Remove from heat when eggs are just set but still moist
7. Season with salt and pepper
8. Serve immediately

# Boiled Eggs
1. Place eggs in a pot
2. Cover with cold water (about 1 inch above eggs)
3. Bring water to a boil
4. Remove from heat and cover pot
5. Let sit: 4-6 minutes for soft-boiled or 10-12 minutes for hard-boiled
6. Transfer eggs to ice water to stop cooking
7. Peel and serve
"""

# Instantiate the ActionPlanningAgent, passing the openai_api_key and the knowledge variable
agent = ActionPlanningAgent(
    openai_api_key = openai_api_key,
    knowledge = knowledge)

# TPrint the agent's response to the following prompt: "One morning I wanted to have scrambled eggs"
prompt = "One morning I wanted to have scrambled eggs"
steps = agent.extract_steps_from_prompt(prompt)
for step in steps:
    print(step)

1. Crack eggs into a bowl
2. Beat eggs with a fork until mixed
3. Heat pan with butter or oil over medium heat
4. Pour egg mixture into pan
5. Stir gently as eggs cook
6. Remove from heat when eggs are just set but still moist
7. Season with salt and pepper
8. Serve immediately
